In [1]:
import json
from N5_retriever import SubjRetriever
from N4_generator import SubjGenerator
import warnings
import os

In [2]:
# Spenderade lång tid på att försöka lösa felmeddelandena... Det fick bli såhär istället.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# Initiera komponenter (körs en gång)
retriever = SubjRetriever()
generator = SubjGenerator()

print("Retriever laddad (FAISS")
print("Generator laddad (GPT-SW3)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: AI-Sweden-Models/gpt-sw3-1.3b-instruct
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Retriever laddad (FAISS
Generator laddad (GPT-SW3)


In [ ]:
# Testfrågor - Verifiera att allt funkar
test_queries = [
    "ska jag jobba med algebra i årskurs 4-6",
    "kunskapskrav geometri årskurs 6?", 
    "vad krävs för E i matte åk 6?",
    "vad är syftet med matematikundervisning?"
]

print("\n" + "="*80)
print("TEST: Kör 4 frågor genom hela pipelinen")
print("="*80)

for i, query in enumerate(test_queries, 1):
    print(f"\nTEST {i}: '{query}'")
    
    # Hämta relevanta chunks
    retrieval_result = retriever.retrieve(query, k=3, min_score=0.35)
    
    # Generera svar
    answer = generator.generate_response(query, retrieval_result)
    
    print(f"🔢 SVAR: {answer}") 
    print("-" * 80)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'no_repeat_ngram_size', 'repetition_penalty', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=170) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEST: Kör 4 frågor genom hela pipelinen

TEST 1: 'ska jag jobba med algebra i årskurs 4-6'
Query: 'ska jag jobba med algebra i årskurs 4-6'
Year hint: 4, Section hint: None


Both `max_new_tokens` (=170) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 SVAR: Ja, du bör definitivt arbeta med algebra under din tid i grundskolan. Algebra är ett viktigt ämne som hjälper dig att förstå hur saker fungerar och hur man löser problem. Det kommer också att hjälpa dig att utveckla dina resonemangsförmågor och kommunikationsförmåga. Dessutom kan det hjälpa dig senare när du går vidare till gymnasiet eller universitetet. Så ja, det är definitivt värt att investera lite tid i att lära sig grunderna i algebra!
--------------------------------------------------------------------------------

TEST 2: 'kunskapskrav geometri årskurs 6?'
Query: 'kunskapskrav geometri årskurs 6?'
Year hint: 6, Section hint: kunskapskrav


Both `max_new_tokens` (=170) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 SVAR: Ja, här följer några exempel på frågor som kan ställas för att kontrollera att eleven har förstått kunskapskraven i geometri i årskurs 5:

    Vad innebär det att ett område är kongruent med en rätvinklig triangel?
    Hur kan man beräkna omkretsen av en kvadrat?
   Vilka är de grundläggande begreppen i area och volym?
     Kan du förklara hur man beräknar arean av en rektangel?
--------------------------------------------------------------------------------

TEST 3: 'vad krävs för E i matte åk 6?'
Query: 'vad krävs för E i matte åk 6?'
Year hint: 6, Section hint: None


Both `max_new_tokens` (=170) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 SVAR: För att få betyget E i matematik måste du ha minst E i alla delprov.
        
        1. Delproven är:
            - Taluppfattning – 5 delprov
            + Taluppfattningsdelprov A, B, C, D, E
            = 7 delprov totalt

        2. För att få ett E i delproven ska du ha lägst E i något eller några av delproverna.
            Om du har lägst E på något delprov behöver du inte göra om det, men du får inte heller högre än E i någon av de andra delarna.
--------------------------------------------------------------------------------

TEST 4: 'vad är syftet med matematikundervisning?'
Query: 'vad är syftet med matematikundervisning?'
Year hint: None, Section hint: None
📄 SVAR: Syftet med matematik är att hjälpa människor att förstå världen omkring dem och att delta i det dagliga livet genom att använda matematik för att lösa problem och fatta beslut. Matematik hjälper också studenter att utveckla sina kritiska tänkande färdigheter, kreativitet och problemlösningsförmåga.
-----

Den här prototypen för en RAG-bot (Retrieval-Augmented Generation) baserad på kursplanen för matematik, hämtat via API från Skolverket. I en slutgiltig version, med alla ämnen  laddade, kan botten vara en assistent för både elever och lärare. 

## Elever

Boten skulle kunna användas av elever som vill ha en djupare förståelse för kursplanerna, t.ex. inför prov, upsatser eller andra examinationer. Chattbotten skulle kunna vara bidragande till att hålla kusplanen levande, istället för att begränsat till bara den där obligatoriska genomgången i början av en termin som man knappt förstod något av. Chattbotten skulle kunna hjälpa till att göra innehållet mer begripligt och tillgängligt för elever som, exempelvis utifrån ålder, inte har förutsättningar att ta sig an informationen på egen hand. Ett etiskt dilemma är att det mycket väl skulle kunna användas för att fuska mer effektivt - "skriv om min upsats så att den bättre matchar kursmålen", etc. Boten skulle behöva noga begränsas för att förhindra detta - men detta, i min erfarenhet, verkar vara svårt även för de stora företagen att implementera utan att det är relativt lätt att kringgå. 

## Lärare

För lärare finns inte samma risk för fusk. För lärare kan botten vara ett hjälpsamt verktyg för att vara säker på att man håller sig till kursplanera och täcker allt centralt innehåll. Boten skulle kunna hjälpa till med förslag om uppgifter eller för lektionsplanering. Med källhänvisningar kan pedagoger också försäkra sig om att informationen de tar del av stämmer. Med samtliga ämnen inladdade i modellen skulle det också finnas möjligheter för lärare att genomföra större projekt, där man har samverkan mellan ämnen och checkar av centralt innehåll i båda (eller fler?) kursplaner. Pedagoger, särskilt i högre åldrar, har ofta djup kunskap om sitt eget ämne, men begränsad kännedom om andra. En gemensam och lättillgänglig kunskapsbas skulle således kunna öppna dörrar för samverkan. 
Ett naturligt steg för breddning i användningsområde ser jag skulle vara att implementera att modellen också används för bedömning. Den har ju trots allt information om allt en elev ska bedömas på, och datorer är ju alltid opartiska, så bedömningen skulle definitivt alltid bli helt korrekt och objektiv! Eller inte. Här blir det snabbt risk för att felaktiga bedömningar görs i en "svart låda" där man inte kan se vad det är som faktiskt utvärderts, även när modellen ombeds motivera sina beslut. Enligt skollagen (3 kap 16§) ska betyg "beslutas av den eller de lärare som bedriver undervisningen vid den tidpunkt när betyg ska sättas", vilket begränsar detta något, men lämnar fortfarande dörren vidöppen för att ta emot "rekommendationer" av AI. Detta skulle i längden kunna leda till "slarv" i bedömningen av uppgifter, där pedagoger litar blint på den bedömning AI ger. Det skulle också kunna leda till att pedagoger inte granskade elevernas uppgifter på samma sätt, vilket innebär att de inte får kännedom om elevens faktiska kunskaper - något man skulle kunna argumentera faktiskt är hela poängen med alla bedömningar fram till gymnasiet. Läraren går då miste om en möjlighet till att anpassa sin undervisning, samt riskerar att missa elever i behov av särskilt stöd (något som måste uppmärksammas och utredas skyndsamt enligt skollagen).